In [1]:
import random
import torch
import os
import numpy as np
import pandas as pd
import polars as pl

In [2]:
INPUT_DIR = '.'

In [3]:
# Set random states.
def set_random_states(random_state):
    # Set various random seeds.
    np.random.seed(random_state)
    pl.set_random_seed(random_state)
    pd.core.common.random_state(random_state)
    random.seed(random_state)
    torch.manual_seed(random_state)
    torch.cuda.manual_seed_all(random_state)
    os.environ["PYTHONHASHSEED"] = str(random_state)
    os.environ["TOKENIZERS_PARALLELISM"] = "false"
    try:
        torch.use_deterministic_algorithms(True)
    except Exception:
        pass
    return random_state

RANDOM_STATE = set_random_states(1618)

In [4]:
def make_aggregated_outputs(input_file, metrics = ['accuracy', 'precision', 'recall', 'f1', 'kappa', 'MCC']):
    dimensions = [
        'informational_vs_involved',
        'non-narrative_vs_narrative',
        'situation-dependent_vs_explicit',
        'non-persuasive_vs_persuasive',
        'non-abstract_vs_abstract',
        'compressed_vs_elaborated'
    ]

    saved_output_dict = {}

    for folder in os.listdir(INPUT_DIR):
        folder_path = os.path.join(INPUT_DIR, folder)

        if not (os.path.isdir(folder_path) and 'outputs' in folder):
            continue

        saved_output_dict[folder] = {
            dim: {metric: [] for metric in metrics}
            for dim in dimensions
        }

        for sub_folder in os.listdir(folder_path):
            sub_path = os.path.join(folder_path, sub_folder)

            if not (os.path.isdir(sub_path) and sub_folder.isdigit()):
                continue

            file_path = os.path.join(sub_path, f"{input_file}.csv")
            df = pd.read_csv(file_path)

            for dimension in dimensions:
                temp_df = df[df['dimension'] == dimension]

                if temp_df.empty:
                    raise ValueError(f"There should be something in the temp_df for the dimension {dimension}.")

                row = temp_df.iloc[0]

                for metric in metrics:
                    saved_output_dict[folder][dimension][metric].append(float(row[metric]))

    temp_dict_all = {}

    for folder, dimensions_dict in saved_output_dict.items():
        series_list = []

        for dimension, metrics_dict in dimensions_dict.items():
            mean_series = pd.Series({
                metric: (sum(values) / len(values)) if values else float('nan')
                for metric, values in metrics_dict.items()
            }, name=dimension)

            series_list.append(mean_series)

        df_folder = pd.concat(series_list, axis=1)
        temp_dict_all[folder] = df_folder

    df_all_folders = pd.concat(temp_dict_all, axis=0)

    df_mean_all = df_all_folders.groupby(level=1).mean()

    return df_all_folders, df_mean_all

In [5]:
all_classif, mean_classif = make_aggregated_outputs('classification_comparison_results_zero_vs_biber')

In [6]:
all_classif

informational_vs_involved  non-narrative_vs_narrative  \
outputsTrain accuracy                    0.654700                    0.559900   
             precision                   0.566366                    0.462048   
             recall                      0.709752                    0.567595   
             f1                          0.629022                    0.508092   
             kappa                       0.313861                    0.118037   
             MCC                         0.321719                    0.120552   
outputsTest  accuracy                    0.648800                    0.538800   
             precision                   0.559632                    0.437551   
             recall                      0.704285                    0.553233   
             f1                          0.622741                    0.487537   
             kappa                       0.302646                    0.079048   
             MCC                         0.310352                    0.081004   
outputsAll   accuracy                    0.657500                    0.544600   
             precision                   0.569344                    0.445345   
             recall                      0.716357                    0.551750   
             f1                          0.633495                    0.491268   
             kappa                       0.319923                    0.088151   
             MCC                         0.328145                    0.090547   

                        situation-dependent_vs_explicit  \
outputsTrain accuracy                          0.604400   
             precision                         0.660052   
             recall                            0.652502   
             f1                                0.655454   
             kappa                             0.189957   
             MCC                               0.190637   
outputsTest  accuracy                          0.598400   
             precision                         0.655396   
             recall                            0.645192   
             f1                                0.649436   
             kappa                             0.178711   
             MCC                               0.179345   
outputsAll   accuracy                          0.600800   
             precision                         0.655918   
             recall                            0.650098   
             f1                                0.652185   
             kappa                             0.183022   
             MCC                               0.183575   

                        non-persuasive_vs_persuasive  \
outputsTrain accuracy                       0.548100   
             precision                      0.394402   
             recall                         0.529802   
             f1                             0.450928   
             kappa                          0.081397   
             MCC                            0.084207   
outputsTest  accuracy                       0.545200   
             precision                      0.395979   
             recall                         0.525661   
             f1                             0.450176   
             kappa                          0.076124   
             MCC                            0.078725   
outputsAll   accuracy                       0.552600   
             precision                      0.395359   
             recall                         0.531462   
             f1                             0.452121   
             kappa                          0.088501   
             MCC                            0.091632   

                        non-abstract_vs_abstract  compressed_vs_elaborated  
outputsTrain accuracy                   0.624000                  0.406300  
             precision                  0.317203                  0.126424  
             recall                     0.644493                  

In [7]:
mean_classif

,informational_vs_involved,non-narrative_vs_narrative,situation-dependent_vs_explicit,non-persuasive_vs_persuasive,non-abstract_vs_abstract,compressed_vs_elaborated
MCC,0.320072,0.097368,0.184519,0.084854,0.202696,-0.169028
accuracy,0.653667,0.547767,0.601200,0.548633,0.615933,0.407400
f1,0.628420,0.495632,0.652358,0.451075,0.414565,0.187128
kappa,0.312143,0.095079,0.183897,0.082008,0.177198,-0.124154
precision,0.565114,0.448314,0.657122,0.395247,0.310364,0.126481
recall,0.710132,0.557526,0.649264,0.528975,0.633221,0.366529


In [8]:
all_contin, mean_contin = make_aggregated_outputs('continuous_comparison_results_zero_vs_biber', ['pearson', 'spearman', 'MSE', 'RMSE', 'MAE'])

In [9]:
all_contin

informational_vs_involved  non-narrative_vs_narrative  \
outputsTrain pearson                    0.397739                    0.073034   
             spearman                   0.387256                    0.125717   
             MSE                        1.204522                    1.853933   
             RMSE                       1.095699                    1.360346   
             MAE                        0.885198                    1.026749   
outputsTest  pearson                    0.376203                    0.046696   
             spearman                   0.370592                    0.087196   
             MSE                        1.247594                    1.906608   
             RMSE                       1.115247                    1.379128   
             MAE                        0.893992                    1.053592   
outputsAll   pearson                    0.393581                    0.055561   
             spearman                   0.389077                    0.106073   
             MSE                        1.212838                    1.888877   
             RMSE                       1.099491                    1.372922   
             MAE                        0.883885                    1.040274   

                       situation-dependent_vs_explicit  \
outputsTrain pearson                          0.155581   
             spearman                         0.244310   
             MSE                              1.688838   
             RMSE                             1.297674   
             MAE                              0.981831   
outputsTest  pearson                          0.150025   
             spearman                         0.242317   
             MSE                              1.699949   
             RMSE                             1.301523   
             MAE                              0.987876   
outputsAll   pearson                          0.159082   
             spearman                         0.256478   
             MSE                              1.681837   
             RMSE                             1.294204   
             MAE                              0.977277   

                       non-persuasive_vs_persuasive  non-abstract_vs_abstract  \
outputsTrain pearson                       0.103648                  0.156290   
             spearman                      0.148336                  0.247749   
             MSE                           1.792703                  1.687419   
             RMSE                          1.337169                  1.295960   
             MAE                           0.997378                  0.941656   
outputsTest  pearson                       0.107482                  0.142200   
             spearman                      0.142916                  0.226217   
             MSE                           1.785036                  1.715599   
             RMSE                          1.333884                  1.307475   
             MAE                           1.003935                  0.955018   
outputsAll   pearson                       0.112184                  0.149135   
             spearman                      0.148782                  0.245275   
             MSE                           1.775633                  1.701731   
             RMSE                          1.330555                  1.302353   
             MAE                           0.992183                  0.948264   

                       compressed_vs_elaborated  
outputsTrain pearson                  -0.046865  
             spearman                 -0.175302  
             MSE                       2.093731  
             RMSE                      1.445232  
             MAE                       1.117494  
outputsTest  pearson                  -0.049421  
             spearman                 -0.177745  
             MSE                       2.098841  
             RMSE                      1.447380  
             MAE

In [10]:
mean_contin

,informational_vs_involved,non-narrative_vs_narrative,situation-dependent_vs_explicit,non-persuasive_vs_persuasive,non-abstract_vs_abstract,compressed_vs_elaborated
MAE,0.887691,1.040205,0.982328,0.997832,0.948313,1.122763
MSE,1.221651,1.883139,1.690208,1.784457,1.701583,2.099849
RMSE,1.103479,1.370799,1.297800,1.333869,1.301929,1.447545
pearson,0.389174,0.058430,0.154896,0.107771,0.149208,-0.049924
spearman,0.382308,0.106329,0.247702,0.146678,0.239747,-0.174730
